In [1]:
%%bash

pip uninstall torch torchvision torchaudio -y -q

pip install torch==2.3.1+cu118 torchvision==0.18.1+cu118 \
--index-url https://download.pytorch.org/whl/cu118 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.6/839.6 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 74.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 44.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 99.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 33.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 14.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.9/142.9 MB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability())

True
Tesla P100-PCIE-16GB
(6, 0)


In [3]:
%%bash

wget "https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/pub.pt" \
"https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/priv.pt" \
"https://huggingface.co/datasets/SprintML/tml26_task1/resolve/main/model.pt" -q

In [33]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import requests
import random
import argparse

from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models import resnet18
import torchvision.transforms as transforms

from sklearn.metrics import roc_curve

In [5]:
# config for cluster file
# BASE = Path(__file__).parent
# PUB_PATH = BASE / "pub.pt"
# PRIV_PATH = BASE / "priv.pt"
# MODEL_PATH = BASE / "model.pt"
# OUTPUT_CSV = BASE / "submission.csv"

# config for online jupyter notebook
BASE = os.path.dirname(os.path.abspath("__file__"))
PUB_PATH = BASE + "/pub.pt"
PRIV_PATH = BASE + "/priv.pt"
MODEL_PATH = BASE + "/model.pt"
OUTPUT_CSV = BASE + "/submission.csv"

BASE_URL = "http://34.63.153.158"   #DONOT CHANGE
API_KEY = "team_IV 5cd96dee6700b068d36678d03d1b1cec"
TASK_ID = "01-mia"  #DONOT CHANGE

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICES = True if torch.cuda.device_count() > 1 else False

BATCH_SIZE = 64

In [6]:
class TaskDataset(Dataset):
    def __init__(self, transform=None):
        self.ids = []
        self.imgs = []
        self.labels = []
        self.transform = transform

    def __getitem__(self, index):
        id_ = self.ids[index]
        img = self.imgs[index]
        if self.transform is not None:
            img = self.transform(img)
        label = self.labels[index]
        return id_, img, label

    def __len__(self):
        return len(self.ids)

In [7]:
class MembershipDataset(TaskDataset):
    def __init__(self, transform=None):
        super().__init__(transform)
        self.membership = []

    def __getitem__(self, index):
        id_, img, label = super().__getitem__(index)
        return id_, img, label, self.membership[index]

In [8]:
# load datasets

def load_dataset(PUB_PATH = PUB_PATH, PRIV_PATH = PRIV_PATH):
    print("Loading datasets...")
    pub_ds = torch.load(PUB_PATH, weights_only=False)
    priv_ds = torch.load(PRIV_PATH, weights_only=False)

    # normalization (same as training)
    MEAN = [0.7406, 0.5331, 0.7059]
    STD = [0.1491, 0.1864, 0.1301]
    
    transform = transforms.Compose([
        transforms.Resize(32),
        transforms.Normalize(mean=MEAN, std=STD),
    ])
    
    pub_ds.transform = transform
    priv_ds.transform = transform

    return pub_ds, priv_ds

In [51]:
def make_loader(dataset, batch_size = 64, shuffle = False, device = DEVICE):
    def collate_fn(batch):
    # unwrap if needed
        if len(batch[0]) == 1:
            batch = [b[0] for b in batch]
    
        ids, imgs, labels, membership = zip(*batch)
    
        return (
            list(ids),
            torch.stack(imgs),
            torch.tensor(labels),
            torch.tensor(membership)
        )
    return DataLoader(dataset, batch_size = batch_size, shuffle = shuffle, collate_fn = collate_fn)

In [27]:
def load_model(MODEL_PATH = MODEL_PATH):
    # load model
    print("Loading model...")
    model = resnet18(weights=None)
    model.conv1 = torch.nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    model.maxpool = torch.nn.Identity()
    model.fc = torch.nn.Linear(512, 9)
    
    # model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu")) # Template code
    model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device(DEVICE)))
    model.eval()
    model.to(DEVICE)
    return model

In [28]:
def extract_outputs(model, loader, device = DEVICE):
    all_ids = []
    all_scores = []

    model.eval()

    with torch.no_grad():
        for ids, imgs, labels, _ in loader:
            imgs = imgs.to(device)

            logits = model(imgs)
            probs = torch.softmax(logits, dim=1)

            scores = probs.max(dim=1).values

            all_ids.extend(ids)
            all_scores.extend(scores.cpu().tolist())

    return all_ids, all_scores

In [12]:
def write_submission(ids, scores, path = OUTPUT_CSV):
    print("Creating random submission...")

    df = pd.DataFrame({
        "id": ids,
        "score": scores
    })

    df.to_csv(path, index=False)
    print("Saved:", OUTPUT_CSV)

## NAVIE ATTACK

In [74]:
def run_naive_attack(model, priv_data, device, batch_size=64):
    model.to(device)
    model.eval()

    def collate_fn(batch):
        ids, imgs, labels, _ = zip(*batch)
        return list(ids), torch.stack(imgs), torch.tensor(labels)

    loader = DataLoader(
        priv_data,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    all_ids = []
    all_scores = []

    with torch.no_grad():
        for id_, img, label in loader:
            img = img.to(device)

            logits = model(img)
            probs = F.softmax(logits, dim=1)

            scores = probs.max(dim=1).values

            all_ids.extend(id_)
            all_scores.extend(scores.cpu().tolist())

    df = pd.DataFrame({'id': all_ids, 'score': all_scores})
    df.to_csv(OUTPUT_CSV, index=False)
    print("Saved:", OUTPUT_CSV)

In [75]:
run_naive_attack(model, priv_ds, device)

NameError: name 'priv_ds' is not defined

## LIRA ATTACK

In [29]:
def compute_signals(model, loader, device = DEVICE):

    all_labels = []

    scores = {
        "max_conf": [],
        "neg_loss": [],
        "neg_entropy": [],
        "margin": []
    }

    with torch.no_grad():
        for _, imgs, labels, membership in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)
            probs = F.softmax(logits, dim=1)

            # ---- SIGNALS ----

            # 1. max confidence
            max_conf = probs.max(dim=1).values

            # 2. negative loss
            loss = F.cross_entropy(logits, labels, reduction='none')
            neg_loss = -loss

            # 3. negative entropy
            entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)
            neg_entropy = -entropy

            # 4. margin (top1 - top2)
            top2 = torch.topk(probs, k=2, dim=1).values
            margin = top2[:, 0] - top2[:, 1]

            # ---- STORE ----
            scores["max_conf"].extend(max_conf.cpu().tolist())
            scores["neg_loss"].extend(neg_loss.cpu().tolist())
            scores["neg_entropy"].extend(neg_entropy.cpu().tolist())
            scores["margin"].extend(margin.cpu().tolist())

            all_labels.extend(membership.cpu().tolist())

    return scores, np.array(all_labels)

In [30]:
def tpr_at_fpr(scores, labels, target_fpr=0.05):
    fpr, tpr, thresholds = roc_curve(labels, scores)

    # find last index where fpr <= target
    idx = np.where(fpr <= target_fpr)[0]

    if len(idx) == 0:
        return 0.0

    return tpr[idx[-1]]

In [31]:
def evaluate_baselines(model, pub_ds, device = DEVICE):

    scores_dict, labels = compute_signals(model, loader, device)

    results = {}

    print("\n=== Baseline Results (TPR @ 5% FPR) ===\n")

    for name, scores in scores_dict.items():
        tpr = tpr_at_fpr(scores, labels)
        results[name] = tpr
        print(f"{name:15s}: {tpr:.4f}")

    best = max(results, key=results.get)
    print(f"\nBest signal: {best} ({results[best]:.4f})")

    return results

In [32]:
pub_ds, _ = load_dataset()
loader = make_loader(pub_ds)
model = load_model()
results = evaluate_baselines(model, pub_ds)

Loading datasets...
Loading model...

=== Baseline Results (TPR @ 5% FPR) ===

max_conf       : 0.0506
neg_loss       : 0.0507
neg_entropy    : 0.0494
margin         : 0.0497

Best signal: neg_loss (0.0507)


In [52]:
def split_shadow(pub_ds, ratio=0.5):
    indices = np.arange(len(pub_ds))
    np.random.shuffle(indices)

    split = int(ratio * len(indices))

    train_idx = indices[:split]
    out_idx = indices[split:]

    shadow_train = Subset(pub_ds, train_idx)
    shadow_out = Subset(pub_ds, out_idx)

    return shadow_train, shadow_out

In [53]:
def train_shadow(model, loader, device, epochs=3):
    model = model.to(device)
    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss()

    for epoch in range(epochs):
        for _, imgs, labels, _ in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model

In [54]:
def extract_features(model, loader, device):
    model.eval()

    X = []
    y = []

    with torch.no_grad():
        for _, imgs, labels, membership in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)
            probs = F.softmax(logits, dim=1)

            # ---- features ----
            max_conf = probs.max(dim=1).values
            loss = F.cross_entropy(logits, labels, reduction='none')
            entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)

            feats = torch.stack([
                max_conf,
                -loss,          # IMPORTANT
                -entropy
            ], dim=1)

            X.append(feats.cpu())
            y.append(membership)

    X = torch.cat(X)
    y = torch.cat(y)

    return X, y

In [55]:
def build_attack_dataset(shadow_model, train_loader, out_loader, device):
    X_in, y_in = extract_features(shadow_model, train_loader, device)
    X_out, y_out = extract_features(shadow_model, out_loader, device)

    X = torch.cat([X_in, X_out])
    y = torch.cat([y_in, y_out])

    return X, y

In [56]:
# 1. split
pub_ds = load_dataset()
shadow_train_ds, shadow_out_ds = split_shadow(pub_ds)

# 2. loaders
train_loader = make_loader(shadow_train_ds, shuffle=True)
out_loader = make_loader(shadow_out_ds, shuffle=False)

# 3. shadow model
shadow_model = load_model()

# 4. train
shadow_model = train_shadow(shadow_model, train_loader, DEVICE)

# 5. build attack dataset
X_attack, y_attack = build_attack_dataset(
    shadow_model,
    train_loader,
    out_loader,
    DEVICE
)

print(X_attack.shape, y_attack.shape)

Loading datasets...
Loading model...


ValueError: too many values to unpack (expected 4)

In [22]:
# create random submission (remove this later or it will rewrite your actual submission)
print("Creating random submission...")
ids = [str(i) for i in priv_ds.ids]

df = pd.DataFrame({
    "id": ids,
    "score": [random.random() for _ in ids]
})

df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

Creating random submission...
Saved: /kaggle/working/submission.csv


In [ ]:
# # submit
# def die(msg):
#     print(msg, file=sys.stderr)
#     sys.exit(1)

# parser = argparse.ArgumentParser(description="Submit a CSV file to the server.")
# args = parser.parse_args()

# submit_path = OUTPUT_CSV

# if not submit_path.exists():
#     die(f"File not found: {submit_path}")

# try:
#     with open(submit_path, "rb") as f:
#         resp = requests.post(
#             f"{BASE_URL}/submit/{TASK_ID}",
#             headers={"X-API-Key": API_KEY},
#             files={"file": (submit_path.name, f, "application/csv")},
#             timeout=(10, 600),
#         )
#     try:
#         body = resp.json()
#     except Exception:
#         body = {"raw_text": resp.text}

#     if resp.status_code == 413:
#         die("Upload rejected: file too large (HTTP 413).")

#     resp.raise_for_status()

#     print("Successfully submitted.")
#     print("Server response:", body)
#     submission_id = body.get("submission_id")
#     if submission_id:
#         print(f"Submission ID: {submission_id}")

# except requests.exceptions.RequestException as e:
#     detail = getattr(e, "response", None)
#     print(f"Submission error: {e}")
#     if detail is not None:
#         try:
#             print("Server response:", detail.json())
#         except Exception:
#             print("Server response (text):", detail.text)
#     sys.exit(1)